# CityLens Baseline Comparison

Use this notebook for the next experiment phase only:

- lock the current `prithvi_rgb_lora` reference results
- run fair satellite backbone comparisons
- summarize metrics into one table
- decide whether street-only and fusion are worth running next

This notebook is intentionally smaller than `colab_citylens_full.ipynb` and calls the existing training entrypoints instead of duplicating the full pipeline.

## Colab setup

Upload this notebook to a fresh Colab runtime, choose a GPU, then run all cells. This notebook mounts Drive, clones the repo, installs the learned-model dependencies, and runs only the comparison-phase experiments.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

REPO_ROOT = Path('/content/CityLens')
REPO_URL = 'https://github.com/wojackbro/GEOBENCH.git'

if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_ROOT)], check=True)
else:
    if (REPO_ROOT / '.git').exists():
        subprocess.run(['git', 'fetch', 'origin'], cwd=str(REPO_ROOT), check=True)
        subprocess.run(['git', 'checkout', 'main'], cwd=str(REPO_ROOT), check=True)
        subprocess.run(['git', 'pull', '--ff-only', 'origin', 'main'], cwd=str(REPO_ROOT), check=True)
    else:
        raise SystemExit(f'Expected a git repo at {REPO_ROOT}, but .git is missing.')

# Auto-detect project root (some Colab runs have nested /content/CityLens/CityLens)
CANDIDATE_ROOTS = [REPO_ROOT, REPO_ROOT / 'CityLens']
PROJECT_ROOT = None
for candidate in CANDIDATE_ROOTS:
    if (candidate / 'evaluate' / 'global_learned' / 'train.py').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise SystemExit('Could not locate evaluate/global_learned/train.py after clone/pull.')

commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=str(REPO_ROOT), text=True).strip()
print('Repo root:', REPO_ROOT)
print('Project root:', PROJECT_ROOT)
print('Commit:', commit)

Repo root: /content/CityLens
Project root: /content/CityLens/CityLens
Commit: 17d706a


In [2]:
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = Path('/content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data')
os.environ['CITYLENS_DATA_ROOT'] = str(DATA_ROOT)
print('CITYLENS_DATA_ROOT =', os.environ['CITYLENS_DATA_ROOT'])

Mounted at /content/drive
CITYLENS_DATA_ROOT = /content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data


In [3]:
required = [
    DATA_ROOT / 'Benchmark',
    DATA_ROOT / 'Results',
    PROJECT_ROOT,
    PROJECT_ROOT / 'evaluate' / 'global_learned' / 'train.py',
]
for path in required:
    print(path, 'exists=' + str(path.exists()))

if not (DATA_ROOT / 'Benchmark').exists():
    raise SystemExit('CityLens data not found in Drive. Reuse the previous full notebook once to prepare the dataset first.')
if not (PROJECT_ROOT / 'evaluate' / 'global_learned' / 'train.py').exists():
    raise SystemExit('Project checkout is missing evaluate/global_learned/train.py. Re-run the repo clone/pull cell.')

/content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data/Benchmark exists=True
/content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data/Results exists=True
/content/CityLens/CityLens exists=True
/content/CityLens/CityLens/evaluate/global_learned/train.py exists=True


In [4]:
from pathlib import Path
import json
import os
import subprocess
import pandas as pd

DATA_ROOT = Path('/content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data')
CITYLENS_ROOT = PROJECT_ROOT
RESULTS_ROOT = DATA_ROOT / 'Results' / 'global_learned'

REFERENCE_MODELS = ['prithvi_rgb_lora', 'dinov2_sat', 'resnet50_sat']
TASK_EPOCHS = {
    'gdp': 20,
    'acc2health': 30,
    'build_height': 30,
    'pop': 5,
}

LOCKED_PRITHVI = {
    'gdp': {'r2': 0.5807730555534363, 'rmse': 331463584.0, 'best_epoch': 14, 'epochs': 20},
    'acc2health': {'r2': 0.39014101028442383, 'rmse': 9.55018424987793, 'best_epoch': 9, 'epochs': 30},
    'build_height': {'r2': 0.8681523203849792, 'rmse': 2.534539222717285, 'best_epoch': 9, 'epochs': 30},
    'pop': {'r2': -0.03239285945892334, 'rmse': 21641.24609375, 'best_epoch': 2, 'epochs': 5},
}

COMMON = {
    'branch': 'satellite',
    'street_model': 'clip_vitb16',
    'batch_size': 8,
    'image_size': 224,
    'lr': 2e-4,
    'backbone_lr': 2e-4,
    'head_lr': 1e-3,
    'weight_decay': 1e-2,
    'seed': 42,
    'lora_r': 8,
    'target_transform': 'log1p',
    'val_frac': 0.1,
    'num_workers': 2,
}

assert DATA_ROOT.exists(), f'Missing data root: {DATA_ROOT}'
assert CITYLENS_ROOT.exists(), f'Missing project root: {CITYLENS_ROOT}'
assert (CITYLENS_ROOT / 'evaluate' / 'global_learned' / 'train.py').exists(), 'Missing evaluate/global_learned/train.py in CITYLENS_ROOT'
print('Project root:', CITYLENS_ROOT)
print('Results root:', RESULTS_ROOT)

Project root: /content/CityLens/CityLens
Results root: /content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data/Results/global_learned


In [5]:
def run_train(task_name: str, satellite_model: str, epochs: int, resume: bool = True, skip_if_done: bool = True):
    cmd = [
        'python', '-m', 'evaluate.global_learned.train',
        '--task_name', task_name,
        '--branch', COMMON['branch'],
        '--satellite_model', satellite_model,
        '--street_model', COMMON['street_model'],
        '--epochs', str(epochs),
        '--batch_size', str(COMMON['batch_size']),
        '--image_size', str(COMMON['image_size']),
        '--lr', str(COMMON['lr']),
        '--backbone_lr', str(COMMON['backbone_lr']),
        '--head_lr', str(COMMON['head_lr']),
        '--weight_decay', str(COMMON['weight_decay']),
        '--seed', str(COMMON['seed']),
        '--lora_r', str(COMMON['lora_r']),
        '--target_transform', COMMON['target_transform'],
        '--val_frac', str(COMMON['val_frac']),
        '--num_workers', str(COMMON['num_workers']),
        '--data_root', str(DATA_ROOT),
    ]
    if resume:
        cmd.append('--resume')
    if skip_if_done:
        cmd.append('--skip_if_done')

    print('\n$', ' '.join(cmd))
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    subprocess.run(cmd, cwd=str(CITYLENS_ROOT), env=env, check=True)


def run_satellite_matrix(tasks, models):
    for task_name in tasks:
        epochs = TASK_EPOCHS[task_name]
        for satellite_model in models:
            run_train(task_name, satellite_model, epochs)


## Run matched satellite baselines

Recommended immediate comparison set:

1. `build_height`: `dinov2_sat`, `resnet50_sat`
2. `gdp`: `dinov2_sat`, `resnet50_sat`
3. `acc2health`: `dinov2_sat`, `resnet50_sat`

Leave `pop` optional unless you want a fairer failure-case comparison.

In [6]:
from pathlib import Path

p = Path("/content/CityLens/CityLens/evaluate/global_learned/models.py")
txt = p.read_text()

old = 'self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)'
new = 'self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0, img_size=224)'

if old in txt and new not in txt:
    txt = txt.replace(old, new)
    p.write_text(txt)
    print("Patched TimmEncoder to force img_size=224")
else:
    print("Patch already applied or line not found")

Patched TimmEncoder to force img_size=224


In [7]:
!rg "create_model\(model_name, pretrained=True, num_classes=0, img_size=224\)" /content/CityLens/CityLens/evaluate/global_learned/models.py

36:        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0, img_size=224)


In [8]:
from pathlib import Path

p = Path("/content/CityLens/CityLens/evaluate/global_learned/utils.py")
txt = p.read_text()

old = "rmse = mean_squared_error(y_true, y_pred, squared=False)"
new = "rmse = float(np.sqrt(mse))"

if old in txt:
    txt = txt.replace(old, new)
    p.write_text(txt)
    print("Patched compute_metrics: rmse now uses sqrt(mse)")
else:
    print("Patch target not found (may already be patched)")

Patched compute_metrics: rmse now uses sqrt(mse)


In [9]:
!rg "img_size=224" /content/CityLens/CityLens/evaluate/global_learned/models.py

36:        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0, img_size=224)


In [10]:
from pathlib import Path

p = Path("/content/CityLens/CityLens/evaluate/global_learned/utils.py")
txt = p.read_text()

old = "rmse = mean_squared_error(y_true, y_pred, squared=False)"
new = "rmse = float(np.sqrt(mse))"

if old in txt:
    txt = txt.replace(old, new)
    p.write_text(txt)
    print("Patched compute_metrics: rmse now uses sqrt(mse)")
else:
    print("Patch target not found (already patched?)")

Patch target not found (already patched?)


In [11]:
!rg "rmse = " /content/CityLens/CityLens/evaluate/global_learned/utils.py

93:    rmse = float(np.sqrt(mse))


In [12]:
from pathlib import Path

p = Path("/content/CityLens/CityLens/evaluate/global_learned/models.py")
txt = p.read_text()

old_block = """        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0, img_size=224)
        self.feature_dim = getattr(self.backbone, "num_features", None) or getattr(self.backbone, "embed_dim", None)
"""

new_block = """        # Only some timm models (e.g., ViTs) accept img_size in constructor.
        # ResNet and others can fail if img_size is passed.
        if "vit" in model_name or "dinov2" in model_name:
            self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0, img_size=224)
        else:
            self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        self.feature_dim = getattr(self.backbone, "num_features", None) or getattr(self.backbone, "embed_dim", None)
"""

if old_block in txt:
    txt = txt.replace(old_block, new_block)
    p.write_text(txt)
    print("Patched TimmEncoder: img_size only for ViT/DINO models.")
else:
    print("Expected block not found; print constructor section manually and patch by hand.")

Patched TimmEncoder: img_size only for ViT/DINO models.


In [13]:
!sed -n '28,50p' /content/CityLens/CityLens/evaluate/global_learned/models.py

        return out
    return out.view(out.size(0), -1)


class TimmEncoder(nn.Module):
    def __init__(self, model_name: str):
        super().__init__()
        self.model_name = model_name
        # Only some timm models (e.g., ViTs) accept img_size in constructor.
        # ResNet and others can fail if img_size is passed.
        if "vit" in model_name or "dinov2" in model_name:
            self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0, img_size=224)
        else:
            self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        self.feature_dim = getattr(self.backbone, "num_features", None) or getattr(self.backbone, "embed_dim", None)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.backbone(x)
        feat = reduce_backbone_output(out)
        if self.feature_dim is None:
            self.feature_dim = feat.size(-1)
        return feat



In [14]:
TASKS_TO_RUN = ['build_height', 'gdp', 'acc2health']
SAT_MODELS_TO_RUN = ['dinov2_sat', 'resnet50_sat']

# Set EXECUTE = True when ready.
EXECUTE = True

if EXECUTE:
    run_satellite_matrix(TASKS_TO_RUN, SAT_MODELS_TO_RUN)
else:
    for task_name in TASKS_TO_RUN:
        for satellite_model in SAT_MODELS_TO_RUN:
            print(task_name, satellite_model, TASK_EPOCHS[task_name])



$ python -m evaluate.global_learned.train --task_name build_height --branch satellite --satellite_model dinov2_sat --street_model clip_vitb16 --epochs 30 --batch_size 8 --image_size 224 --lr 0.0002 --backbone_lr 0.0002 --head_lr 0.001 --weight_decay 0.01 --seed 42 --lora_r 8 --target_transform log1p --val_frac 0.1 --num_workers 2 --data_root /content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data --resume --skip_if_done

$ python -m evaluate.global_learned.train --task_name build_height --branch satellite --satellite_model resnet50_sat --street_model clip_vitb16 --epochs 30 --batch_size 8 --image_size 224 --lr 0.0002 --backbone_lr 0.0002 --head_lr 0.001 --weight_decay 0.01 --seed 42 --lora_r 8 --target_transform log1p --val_frac 0.1 --num_workers 2 --data_root /content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data --resume --skip_if_done

$ python -m evaluate.global_learned.train --task_name gdp --branch satellite --satellite_model dinov2_sat --street_model clip_vitb16 --epoc

In [15]:
def find_metrics(task_name: str, satellite_model: str, epochs: int):
    task_root = RESULTS_ROOT / task_name
    if not task_root.exists():
        return None
    pattern = f"{task_name}-satellite-{satellite_model}-clip_vitb16-bs8-ep{epochs}-lr0.0002-blr0.0002-hlr0.001-ttlog1p-seed42"
    exp_dir = task_root / pattern
    metrics_path = exp_dir / 'metrics.json'
    if not metrics_path.exists():
        return None
    payload = json.loads(metrics_path.read_text())
    return {
        'task': task_name,
        'model': satellite_model,
        'epochs': epochs,
        'best_epoch': payload.get('best_epoch'),
        'r2': payload.get('r2'),
        'rmse': payload.get('rmse'),
        'mae': payload.get('mae'),
        'mse': payload.get('mse'),
        'path': str(exp_dir),
    }


rows = []
for task_name, ref in LOCKED_PRITHVI.items():
    rows.append({
        'task': task_name,
        'model': 'prithvi_rgb_lora',
        'epochs': ref['epochs'],
        'best_epoch': ref['best_epoch'],
        'r2': ref['r2'],
        'rmse': ref['rmse'],
        'mae': None,
        'mse': None,
        'path': 'docs/PRITHVI_SATELLITE_REFERENCE.md',
    })

for task_name in TASK_EPOCHS:
    for satellite_model in ['dinov2_sat', 'resnet50_sat']:
        row = find_metrics(task_name, satellite_model, TASK_EPOCHS[task_name])
        if row is not None:
            rows.append(row)

df = pd.DataFrame(rows)
df.sort_values(['task', 'model']) if not df.empty else df

,task,model,epochs,best_epoch,r2,rmse,mae,mse,path
6,acc2health,dinov2_sat,30,20,9.849167e-02,1.161133e+01,8.774560e+00,1.348230e+02,/content/drive/MyDrive/GeoBench/CityLens-Data/...
1,acc2health,prithvi_rgb_lora,30,9,3.901410e-01,9.550184e+00,NaN,NaN,docs/PRITHVI_SATELLITE_REFERENCE.md
7,acc2health,resnet50_sat,30,22,2.124074e-01,1.085295e+01,7.204183e+00,1.177866e+02,/content/drive/MyDrive/GeoBench/CityLens-Data/...
8,build_height,dinov2_sat,30,18,6.790898e-01,3.954163e+00,2.812445e+00,1.563540e+01,/content/drive/MyDrive/GeoBench/CityLens-Data/...
2,build_height,prithvi_rgb_lora,30,9,8.681523e-01,2.534539e+00,NaN,NaN,docs/PRITHVI_SATELLITE_REFERENCE.md
9,build_height,resnet50_sat,30,26,8.004348e-01,3.118209e+00,2.280582e+00,9.723226e+00,/content/drive/MyDrive/GeoBench/CityLens-Data/...
4,gdp,dinov2_sat,20,19,4.534916e-01,3.784510e+08,2.328081e+08,1.432252e+17,/content/drive/MyDrive/GeoBench/CityLens-Data/...
0,gdp,prithvi_rgb_lora,20,14,5.807731e-01,3.314636e+08,NaN,NaN,docs/PRITHVI_SATELLITE_REFERENCE.md
5,gdp,resnet50_sat,20,1,-8.620295e+12,1.503046e+15,7.372829e+13,2.259147e+30,/content/drive/MyDrive/GeoBench/CityLens-Data/...
3,pop,prithvi_rgb_lora,5,2,-3.239286e-02,2.164125e+04,NaN,NaN,docs/PRITHVI_SATELLITE_REFERENCE.md


In [16]:
if df.empty:
    print('No baseline comparison rows found yet.')
else:
    pivot = df.pivot_table(index='task', columns='model', values='r2', aggfunc='first')
    display(pivot)

    wins = 0
    considered = 0
    for task_name in ['build_height', 'gdp', 'acc2health']:
        if task_name not in pivot.index:
            continue
        if 'prithvi_rgb_lora' not in pivot.columns:
            continue
        other_values = [pivot.loc[task_name, c] for c in pivot.columns if c != 'prithvi_rgb_lora' and pd.notna(pivot.loc[task_name, c])]
        if not other_values:
            continue
        considered += 1
        if pivot.loc[task_name, 'prithvi_rgb_lora'] > max(other_values):
            wins += 1

    if considered == 0:
        print('Need baseline runs before making the modality decision.')
    elif wins >= 2:
        print('Recommendation: proceed to street-only clip_vitb16 and late fusion next.')
    else:
        print('Recommendation: stop and frame this as a smaller satellite empirical study unless stronger baseline results arrive.')


model,dinov2_sat,prithvi_rgb_lora,resnet50_sat
task,,,
acc2health,0.098492,0.390141,2.124074e-01
build_height,0.679090,0.868152,8.004348e-01
gdp,0.453492,0.580773,-8.620295e+12
pop,NaN,-0.032393,NaN


Recommendation: proceed to street-only clip_vitb16 and late fusion next.


In [17]:
from pathlib import Path
import json
import pandas as pd

root = Path("/content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data/Results/global_learned")

tasks = ["gdp", "acc2health", "build_height", "pop"]
models = ["prithvi_rgb_lora", "dinov2_sat", "resnet50_sat"]

# Epoch budgets you used
task_epochs = {"gdp": 20, "acc2health": 30, "build_height": 30, "pop": 5}

rows = []
for t in tasks:
    for m in models:
        exp = root / t / f"{t}-satellite-{m}-clip_vitb16-bs8-ep{task_epochs[t]}-lr0.0002-blr0.0002-hlr0.001-ttlog1p-seed42"
        mp = exp / "metrics.json"
        if mp.exists():
            d = json.loads(mp.read_text())
            rows.append({
                "task": t, "model": m,
                "best_epoch": d.get("best_epoch"),
                "r2": d.get("r2"),
                "rmse": d.get("rmse"),
                "mae": d.get("mae"),
                "path": str(exp)
            })
        else:
            rows.append({
                "task": t, "model": m,
                "best_epoch": None, "r2": None, "rmse": None, "mae": None,
                "path": f"MISSING: {exp}"
            })

df = pd.DataFrame(rows)
print(df.sort_values(["task","model"]).to_string(index=False))

print("\nR2 Pivot:")
print(df.pivot_table(index="task", columns="model", values="r2", aggfunc="first"))

        task            model  best_epoch            r2         rmse          mae                                                                                                                                                                                                    path
  acc2health       dinov2_sat        20.0  9.849167e-02 1.161133e+01 8.774560e+00           /content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data/Results/global_learned/acc2health/acc2health-satellite-dinov2_sat-clip_vitb16-bs8-ep30-lr0.0002-blr0.0002-hlr0.001-ttlog1p-seed42
  acc2health prithvi_rgb_lora         9.0  3.901410e-01 9.550184e+00 7.191029e+00     /content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data/Results/global_learned/acc2health/acc2health-satellite-prithvi_rgb_lora-clip_vitb16-bs8-ep30-lr0.0002-blr0.0002-hlr0.001-ttlog1p-seed42
  acc2health     resnet50_sat        22.0  2.124074e-01 1.085295e+01 7.204183e+00         /content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data/Resu